# RAG Step 2 — Indexing

This notebook walks through `scripts/index_statements.py` one concept at a time.

**The RAG pipeline so far:**
```
Step 1 (done) → statements/  (raw data)
Step 2 (this) → ChromaDB     (vectors + metadata)
Step 3 (next) → Retriever    (query → top-k chunks)
Step 4 (next) → Chain        (chunks + query → GPT-4 → answer)
```

**What we build here:**
1. Understand what a chunk is and why chunking matters
2. Convert chunks to embeddings (vectors)
3. Store vectors in ChromaDB with metadata
4. Run a test similarity search to verify it works

## 0. Setup

Three packages doing three distinct jobs:
- `openai` — calls the embedding API (`text-embedding-ada-002`)
- `chromadb` — local vector database (stores and searches vectors)
- `python-dotenv` — reads your `.env` file so the API key isn't hardcoded

In [1]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
import chromadb
from openai import OpenAI

load_dotenv()  # reads .env → injects OPENAI_API_KEY into os.environ

ROOT = Path("..")
STATEMENTS_DIR = ROOT / "data" / "statements"
CARDS_JSON     = ROOT / "data" / "cards.json"
CHROMA_DIR     = ROOT / "data" / "chroma"

openai_client = OpenAI()
print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

API key loaded: True


## 1. What is a chunk?

A **chunk** is the atomic unit of retrieval — the smallest piece of text that gets embedded and stored.

The core design question: *what should one chunk answer?*

We use two chunk types:
- **Spend summary** — what a user actually spent and earned, per card per category per month
- **Card rules** — what a card promises to reward (from `cards.json`)

Retrieving both lets the LLM compare *actual* vs *potential* earnings without hallucinating card terms.

Let's look at real examples of each.

In [2]:
# Load one statement and one card to build example chunks manually
alice_dec = json.loads((STATEMENTS_DIR / "alice" / "2025-12.json").read_text())
cards      = json.loads(CARDS_JSON.read_text())
card_configs = {c["id"]: c["reward_config"] for c in cards if "reward_config" in c}

print("Statement keys:", list(alice_dec.keys()))
print("Cards with reward_config:", list(card_configs.keys()))

Statement keys: ['user_id', 'name', 'month', 'cards', 'transactions', 'reward_postings', 'summary', 'rewards_summary']
Cards with reward_config: ['chase-sapphire-preferred', 'amex-gold', 'citi-double-cash', 'discover-it-cash-back']


In [3]:
# --- Build a spend summary chunk manually ---
# The chunk is a prose sentence. Why prose? Because the embedding model
# was trained on natural language — it understands "dining" better as a word
# in a sentence than as a JSON key.

user_id  = alice_dec["user_id"]
name     = alice_dec["name"]
month    = alice_dec["month"]

# Compute spend per (card, category) from the transactions list
spend_by = {}
for txn in alice_dec["transactions"]:
    key = (txn["card_id"], txn["category"])
    if key not in spend_by:
        spend_by[key] = {"amount": 0.0, "count": 0}
    spend_by[key]["amount"] += txn["amount"]
    spend_by[key]["count"]  += 1

# Pick one (card, category) to show
card_id  = "citi-double-cash"
category = "dining"
s        = spend_by[(card_id, category)]
cat_data = alice_dec["rewards_summary"][card_id]["by_category"][category]

cashback = cat_data["cashback_usd"]
rate     = cat_data["rate"]
chunk_text = (
    f"{name} ({user_id}) spent ${s['amount']:.2f} on {category} "
    f"using {card_id} in {month} across {s['count']} transaction(s) "
    f"and earned ${cashback:.2f} cashback at {rate}."
)

print("SPEND SUMMARY CHUNK:")
print(chunk_text)
print()
print("This chunk answers: 'How much did Alice earn on dining with Citi in Dec 2025?'")

SPEND SUMMARY CHUNK:
Alice Chen (alice) spent $293.71 on dining using citi-double-cash in 2025-12 across 11 transaction(s) and earned $5.87 cashback at 2%.

This chunk answers: 'How much did Alice earn on dining with Citi in Dec 2025?'


In [4]:
# --- Build a card rules chunk manually ---
# This chunk encodes the card's reward terms as prose.
# At query time, retrieving this alongside spend summaries gives the LLM
# the grounding it needs to say "you could be earning more".

card    = next(c for c in cards if c["id"] == "amex-gold")
cfg     = card["reward_config"]
perks   = card.get("perks", [])

lines = [f"{card['name']} (annual fee: ${card['annual_fee']})."]
lines.append(f"Earns {cfg['currency']} points, valued at {cfg['point_value_cents']}¢ per point.")
for cat, rate in cfg["category_rates"].items():
    lines.append(f"  {rate}x {cfg['currency']} on {cat}.")
lines.append("Key perks: " + "; ".join(perks[:3]) + ".")

card_rules_chunk = " ".join(lines)

print("CARD RULES CHUNK:")
print(card_rules_chunk)
print()
print("This chunk answers: 'What does Amex Gold reward and at what rates?'")

CARD RULES CHUNK:
American Express Gold (annual fee: $250). Earns Membership Rewards points, valued at 1.0¢ per point.   4x Membership Rewards on dining.   4x Membership Rewards on groceries.   3x Membership Rewards on travel.   1x Membership Rewards on other. Key perks: $120 dining credit (Grubhub, Cheesecake Factory, and more); $120 Uber Cash annually; No foreign transaction fees.

This chunk answers: 'What does Amex Gold reward and at what rates?'


## 2. What is an embedding?

An embedding converts a text string into a **vector** — a list of floats that encodes meaning as geometry.

- `text-embedding-ada-002` always produces a **1,536-dimensional** vector
- Semantically similar texts produce vectors that are **close** in that space (small cosine distance)
- Unrelated texts produce vectors that are **far apart**

This is what powers semantic search: instead of matching keywords, we match *meaning*.

Let's embed two similar sentences and one unrelated one, then measure the distances.

In [5]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

test_sentences = [
    "Alice spent money eating at restaurants this month.",       # similar to query
    "User dining expenditure at food establishments in December.", # same meaning, different words
    "The annual fee for this credit card is ninety-five dollars.",  # unrelated
]

response = openai_client.embeddings.create(
    model="text-embedding-ada-002",
    input=test_sentences,
)
vecs = [item.embedding for item in response.data]

print(f"Vector dimensions: {len(vecs[0])}")
print(f"First 5 values of vec[0]: {vecs[0][:5]}")
print()
print("Cosine similarities (1.0 = identical, 0.0 = orthogonal):")
print(f"  Sentence 0 vs 1 (same meaning):  {cosine_similarity(vecs[0], vecs[1]):.4f}")
print(f"  Sentence 0 vs 2 (unrelated):     {cosine_similarity(vecs[0], vecs[2]):.4f}")

Vector dimensions: 1536
First 5 values of vec[0]: [0.0016538597410544753, -0.01580444909632206, 0.009574721567332745, -0.027667509391903877, 0.0011149980127811432]

Cosine similarities (1.0 = identical, 0.0 = orthogonal):
  Sentence 0 vs 1 (same meaning):  0.8741
  Sentence 0 vs 2 (unrelated):     0.7690


## 3. Build all chunks

Now we scale up: build all chunks from all 18 statement files + all 4 cards.

Each chunk is a dict with three fields:
- `id` — unique string key (used for upsert deduplication)
- `text` — the prose sentence that gets embedded
- `metadata` — structured tags for filtering at query time

In [6]:
def build_spend_chunks(statement, card_configs):
    chunks = []
    user_id = statement["user_id"]
    name    = statement["name"]
    month   = statement["month"]

    spend_by = {}
    for txn in statement.get("transactions", []):
        key = (txn["card_id"], txn["category"])
        if key not in spend_by:
            spend_by[key] = {"amount": 0.0, "count": 0}
        spend_by[key]["amount"] += txn["amount"]
        spend_by[key]["count"]  += 1

    for card_id, card_data in statement.get("rewards_summary", {}).items():
        reward_type = card_data.get("type")
        for category, cat_data in card_data.get("by_category", {}).items():
            s      = spend_by.get((card_id, category), {})
            spend  = round(s.get("amount", 0), 2)
            count  = s.get("count", 0)
            rate   = cat_data.get("rate", "?")

            if reward_type == "cashback":
                cashback = cat_data.get("cashback_usd", 0)
                earned   = f"earned ${cashback:.2f} cashback at {rate}"
            else:
                points = cat_data.get("points", 0)
                pv     = card_configs.get(card_id, {}).get("point_value_cents", 1.0)
                est    = round(points * pv / 100, 2)
                earned = f"earned {points:,} points at {rate} (≈${est:.2f})"

            text = (
                f"{name} ({user_id}) spent ${spend:.2f} on {category} "
                f"using {card_id} in {month} across {count} transaction(s) "
                f"and {earned}."
            )
            chunks.append({
                "id": f"{user_id}__{card_id}__{category}__{month}",
                "text": text,
                "metadata": {
                    "chunk_type": "spend_summary",
                    "user_id":    user_id,
                    "card_id":    card_id,
                    "category":   category,
                    "month":      month,
                    "spend_usd":  spend,
                },
            })
    return chunks


def build_card_rule_chunks(cards):
    chunks = []
    for card in cards:
        cfg = card.get("reward_config")
        if not cfg:
            continue
        lines = [f"{card['name']} (annual fee: ${card['annual_fee']})."]
        if cfg["type"] == "points":
            pv = cfg["point_value_cents"]
            lines.append(f"Earns {cfg['currency']} points, valued at {pv}¢ per point.")
            for cat, rate in cfg["category_rates"].items():
                lines.append(f"  {rate}x {cfg['currency']} on {cat}.")
            sb = cfg.get("streaming_bonus")
            if sb:
                lines.append(f"  {sb['rate']}x on streaming: {', '.join(sb['merchants'])}.")
        elif cfg["type"] == "cashback":
            if "flat_rate" in cfg:
                lines.append(f"Earns {int(cfg['flat_rate']*100)}% cashback on all purchases.")
            elif "rotating_schedule" in cfg:
                cap  = cfg.get("quarterly_cap", 1500)
                base = int(cfg["base_rate"] * 100)
                rot  = int(cfg["rotating_rate"] * 100)
                lines.append(f"Earns {rot}% on rotating quarterly categories (cap ${cap:,.0f}/qtr), {base}% otherwise.")
                for qtr, cats in cfg["rotating_schedule"].items():
                    for cat, merchants in cats.items():
                        if merchants:
                            lines.append(f"  {qtr}: {rot}% on {cat} at {', '.join(merchants)}.")
                        else:
                            lines.append(f"  {qtr}: {rot}% on all {cat}.")
        perks = card.get("perks", [])
        if perks:
            lines.append("Key perks: " + "; ".join(perks[:3]) + ".")
        chunks.append({
            "id":   f"card_rules__{card['id']}",
            "text": " ".join(lines),
            "metadata": {"chunk_type": "card_rules", "card_id": card["id"]},
        })
    return chunks


# Build everything
all_statements = [json.loads(p.read_text()) for p in sorted(STATEMENTS_DIR.rglob("*.json"))]
spend_chunks   = []
for stmt in all_statements:
    spend_chunks.extend(build_spend_chunks(stmt, card_configs))

rule_chunks = build_card_rule_chunks(cards)
all_chunks  = rule_chunks + spend_chunks

print(f"Card rule chunks : {len(rule_chunks)}")
print(f"Spend chunks     : {len(spend_chunks)}")
print(f"Total            : {len(all_chunks)}")
print()
print("Sample spend chunk:")
print(spend_chunks[0]["text"])
print("Metadata:", spend_chunks[0]["metadata"])

Card rule chunks : 4
Spend chunks     : 114
Total            : 118

Sample spend chunk:
Alice Chen (alice) spent $481.12 on shopping using citi-double-cash in 2025-12 across 5 transaction(s) and earned $9.63 cashback at 2%.
Metadata: {'chunk_type': 'spend_summary', 'user_id': 'alice', 'card_id': 'citi-double-cash', 'category': 'shopping', 'month': '2025-12', 'spend_usd': 481.12}


## 4. Embed all chunks

One API call embeds all 118 chunks in a single batch.

**Why batch?** The Embeddings endpoint accepts up to 2,048 texts at once. Batching is faster and costs the same as individual calls — tokens are tokens.

After this cell you'll have 118 vectors, each a list of 1,536 floats.

In [7]:
texts = [c["text"] for c in all_chunks]

print(f"Embedding {len(texts)} chunks...")
response = openai_client.embeddings.create(
    model="text-embedding-ada-002",
    input=texts,
)
vectors = [item.embedding for item in response.data]

print(f"Done. Each vector has {len(vectors[0])} dimensions.")
print(f"Total vectors: {len(vectors)}")

Embedding 118 chunks...
Done. Each vector has 1536 dimensions.
Total vectors: 118


## 5. Store in ChromaDB

ChromaDB is a local vector database. Think of it like SQLite, but instead of rows it stores vectors.

Key concepts:
- **Collection** — like a table; one collection for all our data
- **`hnsw:space=cosine`** — the distance metric (angle between vectors, not magnitude)
- **`upsert`** — idempotent: re-running this notebook won't create duplicates (matched by `id`)
- **Metadata** — tags stored alongside each vector; used at query time to filter by `user_id`, `chunk_type`, etc.

In [8]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collection = chroma_client.get_or_create_collection(
    name="cardwise",
    metadata={"hnsw:space": "cosine"},
)

collection.upsert(
    ids        = [c["id"]       for c in all_chunks],
    embeddings = vectors,
    documents  = texts,
    metadatas  = [c["metadata"] for c in all_chunks],
)

print(f"Total vectors in collection: {collection.count()}")

Total vectors in collection: 118


## 6. Test: Semantic Search (Retrieval Preview)

Before we build the full RAG chain, let's verify that the index actually retrieves *relevant* chunks for a natural language query.

This is pure retrieval — no LLM, no answer generation. Just:
1. Embed the query
2. Find the closest vectors in ChromaDB (filtered to a specific user)
3. Print the matching chunk texts

If the chunks make sense given the query, the indexing is correct.

In [9]:
def retrieve(query: str, user_id: str, n_results: int = 5):
    """Embed query → similarity search → return top-k chunks for this user."""
    query_vec = openai_client.embeddings.create(
        model="text-embedding-ada-002",
        input=[query],
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=n_results,
        where={"user_id": user_id},   # metadata filter — only this user's chunks
        include=["documents", "metadatas", "distances"],
    )

    docs      = results["documents"][0]
    metas     = results["metadatas"][0]
    distances = results["distances"][0]

    print(f'Query : "{query}"')
    print(f"User  : {user_id}")
    print("-" * 60)
    for doc, meta, dist in zip(docs, metas, distances):
        print(f"[distance={dist:.4f}] {doc}")
        print()

retrieve("Am I getting good value from dining on my cards?", user_id="alice")

Query : "Am I getting good value from dining on my cards?"
User  : alice
------------------------------------------------------------
[distance=0.1788] Alice Chen (alice) spent $205.91 on dining using citi-double-cash in 2026-04 across 8 transaction(s) and earned $4.11 cashback at 2%.

[distance=0.1790] Alice Chen (alice) spent $10.03 on dining using chase-sapphire-preferred in 2026-04 across 1 transaction(s) and earned 30 points at 3x (≈$0.38).

[distance=0.1794] Alice Chen (alice) spent $293.71 on dining using citi-double-cash in 2025-12 across 11 transaction(s) and earned $5.87 cashback at 2%.

[distance=0.1804] Alice Chen (alice) spent $36.25 on dining using chase-sapphire-preferred in 2026-01 across 1 transaction(s) and earned 109 points at 3x (≈$1.36).

[distance=0.1805] Alice Chen (alice) spent $287.08 on dining using citi-double-cash in 2026-01 across 7 transaction(s) and earned $5.74 cashback at 2%.



In [10]:
# Try a different query — notice how different chunks surface
retrieve("Should I be using a different card for gas purchases?", user_id="alice")

Query : "Should I be using a different card for gas purchases?"
User  : alice
------------------------------------------------------------
[distance=0.1960] Alice Chen (alice) spent $44.34 on gas using chase-sapphire-preferred in 2025-12 across 1 transaction(s) and earned 44 points at 1x (≈$0.55).

[distance=0.1965] Alice Chen (alice) spent $52.13 on gas using chase-sapphire-preferred in 2026-03 across 1 transaction(s) and earned 52 points at 1x (≈$0.65).

[distance=0.2015] Alice Chen (alice) spent $212.33 on gas using citi-double-cash in 2026-04 across 4 transaction(s) and earned $4.25 cashback at 2%.

[distance=0.2016] Alice Chen (alice) spent $72.07 on gas using citi-double-cash in 2026-01 across 1 transaction(s) and earned $1.44 cashback at 2%.

[distance=0.2033] Alice Chen (alice) spent $305.29 on gas using citi-double-cash in 2026-05 across 6 transaction(s) and earned $6.12 cashback at 2%.



In [11]:
# Card rules chunks are global (no user_id). Query them separately.
def retrieve_card_rules(query: str, n_results: int = 2):
    query_vec = openai_client.embeddings.create(
        model="text-embedding-ada-002",
        input=[query],
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=n_results,
        where={"chunk_type": "card_rules"},
        include=["documents", "distances"],
    )

    print(f'Query : "{query}"')
    print("-" * 60)
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"[distance={dist:.4f}] {doc}")
        print()

retrieve_card_rules("Which card gives the most rewards on groceries?")

Query : "Which card gives the most rewards on groceries?"
------------------------------------------------------------
[distance=0.1814] American Express Gold (annual fee: $250). Earns Membership Rewards points, valued at 1.0¢ per point.   4x Membership Rewards on dining.   4x Membership Rewards on groceries.   3x Membership Rewards on travel.   1x Membership Rewards on other. Key perks: $120 dining credit (Grubhub, Cheesecake Factory, and more); $120 Uber Cash annually; No foreign transaction fees.

[distance=0.1823] Discover it Cash Back (annual fee: $0). Earns 5% on rotating quarterly categories (cap $1,500/qtr), 1% otherwise.   2025-Q4: 5% on shopping at Amazon, Walmart, Target.   2026-Q1: 5% on all groceries.   2026-Q1: 5% on entertainment at Netflix, Spotify, Hulu, Disney+, Apple TV+, HBO Max.   2026-Q2: 5% on all gas. Key perks: No annual fee; Cashback Match — Discover matches all cashback in year one; Free FICO credit score on every statement.



## Summary

What we built in this notebook:

| Step | Concept | Tool |
|---|---|---|
| Chunking | Break data into retrieval units | Python |
| Embedding | Convert text → 1,536-dim vector | `text-embedding-ada-002` |
| Storage | Persist vectors + metadata | ChromaDB |
| Retrieval | Embed query → cosine similarity search | ChromaDB |

**What's missing:** an LLM. Right now we retrieve relevant chunks but don't *answer* the question. That's Step 3 — the retriever + chain, where retrieved context is passed to GPT-4 to generate a grounded answer.

**Key insight from the distance scores:** a lower distance means the chunk is semantically closer to the query. You'll notice spending chunks for the same category cluster together, and card rule chunks surface when the query is about card terms — even without any keyword matching.